In [2]:
#Define the project's root directory. This is important because we are not yet structuring it as a Python package; 
# it is currently just a collection of scripts.
import os, sys
project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

In [3]:
import numpy as np
from config import load_config
config = load_config()

In [ ]:
#build training dataset
from src.data_pipeline.training.build_training_dataset import build_training_dataset
traing_ds = build_training_dataset(config)

In [ ]:
#build global dataset from scratch
import pandas as pd
from pathlib import Path
from src.data_pipeline.worldwide.build_global_dataset import build_global_dataset, assemble_global_dataset, process_climate, process_population

# The land-use data is available only through 2019 (inclusive). 
# For 2020 and subsequent years, we assume that the land-use values 
# remain unchanged from their 2019 levels.

# Build global dataset up to 2019
for year in np.arange(2015, 2020):
    build_global_dataset(config, year)

resolution = 0.25
lulc_year = 2019
paths = config["paths"]
fname_land_use = Path(paths["processed_data"]["land_use_worldwide"], f"{lulc_year}_land_use_worldwide_res_{resolution}_deg.csv")
df_land_use = pd.read_csv(fname_land_use)
# for following years 2020-...
for year in np.arange(2020, 2025):
    #extract global climate and population datasets datasets from 
    df_climate = process_climate(config, year)
    df_population = process_population(config, year)
    
    #copy lulc 2019 to the current year
    df_lu = df_land_use.copy()
    df_lu["year"] = year
    fname_land_use_new = Path(paths["processed_data"]["land_use_worldwide"], f"{year}_land_use_worldwide_res_{resolution}_deg.csv")
    df_lu.to_csv(fname_land_use_new, sep=',', index=False, decimal='.') 

    #assemble dataset
    global_df = assemble_global_dataset(df_climate, df_lu, df_population)

    #save dataset
    fname_global = Path(paths["processed_data"]["global_dataset"], f"{year}_global_clima_lulc_pop_res_{resolution}_deg.csv")
    global_df.to_csv(fname_global, sep=',', index=False, decimal='.')



In [ ]:
# Assemble global dataset from already prepared sub-sets (climate, land use, population), if they were prepared beforehand
from src.data_pipeline.worldwide.build_global_dataset import assemble_global_dataset, process_climate

years = np.arange(2015, 2025)
resolution = 0.25
paths = config["paths"]

for year in years:
    lulc_year = min(year, 2019)
    
    fname_climate = Path(paths["processed_data"]["climate_worldwide"], f"{year}_climate_worldwide_res_{resolution}_deg.csv")
    df_climate = pd.read_csv(fname_climate)
    
    fname_land_use = Path(paths["processed_data"]["land_use_worldwide"], f"{lulc_year}_land_use_worldwide_res_{resolution}_deg.csv")
    df_land_use = pd.read_csv(fname_land_use)
    
    fname_population = Path(paths["processed_data"]["population_worldwide"], f"{year}_population_worldwide_res_{resolution}_deg.csv")
    df_population = pd.read_csv(fname_population)

    global_df = assemble_global_dataset(df_climate, df_land_use, df_population)

    fname_global = Path(paths["processed_data"]["global_dataset"], f"{year}_global_clima_lulc_pop_res_{resolution}_deg.csv")
    global_df.to_csv(fname_global, sep=',', index=False, decimal='.')

In [ ]:
#prepare gmod dataset
#at the moment, creation of the GMOD depends on the global dataset,
#before running this extraction, make sure global dataset is already prepared
from src.data_pipeline.gmod import build_gmod_dataset

specie = "aegypti" # define specie

#define time frame for incremental learning
gmod_year_start = 2015 
gmod_year_end = 2015

#prepare gmod dataset. it will be savd according to pathes defined in the CONFIG
build_gmod_dataset(gmod_year_start, gmod_year_end, config)

In [ ]:
from src.data_pipeline.worldwide.build_global_dataset import build_global_dataset
import numpy as np

specie = "aegypti" # define specie
for year in np.arange(2013, 2015):
    build_global_dataset(config, year)
    